<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
Memory-Systeme
</b></font> </br></p>

---

**Beitrag zum Leitprojekt:** Memory ist kontrolliertes Kontextmanagement über eine einzelne Sitzung hinaus — der Meeting- & Research-Briefing-Agent kann frühere Fragen, bereits geprüfte Protokolle oder wiederkehrende Themen einer Nutzerin für späteres **Planen** nutzen. Wichtig ist die Abgrenzung: Memory ist kontrolliert und zweckgebunden (Projektwissen, Verlauf), kein offenes Sammeln beliebiger Informationen.

 *Bekannt aus GenAI, hier als Agentenbaustein:* Conversation-Memory-Patterns sind aus GenAI bekannt; neu ist die Einordnung als kontrollierte, zweckgebundene Schicht neben State und Checkpointing.

In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"]    = "M19-Memory-Systeme"
os.environ["LANGSMITH_ENDPOINT"]   = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS

In [ ]:
#@title 📦 Installationen{ display-mode: "form" }
install_packages([("langgraph-checkpoint-sqlite", "langgraph.checkpoint.sqlite")])

<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [Trimming und Summary](https://editor.p5js.org/ralf.bendig.rb/full/Xn-7mkuSM)
- [Caching](https://editor.p5js.org/ralf.bendig.rb/full/3YjAbYaLb)
- [Kontextfenster](https://editor.p5js.org/ralf.bendig.rb/full/tLnUgyZRK)



# 1 | Warum brauchen Agenten Memory?
---


M17 hat gezeigt, wie Checkpointing eine laufende Research-Session wiederaufnehmbar macht. M19 zieht die Grenze zur dauerhaften Memory-Schicht: Mara möchte wiederkehrende Briefing-Präferenzen wie Ausgabeformat, Interessengebiet und Ausschlussregeln behalten, aber keine flüchtigen Session-Details ungeprüft speichern.

Ohne diese Trennung wächst Memory zur unkontrollierten Ablage. Gute Memory-Systeme entscheiden deshalb explizit, was nur Session-State bleibt, was dauerhaft gespeichert werden darf und wie gespeicherte Informationen pro Nutzer oder Projekt getrennt werden.

| Problem | Lösung |
|---------|--------|
| Gesprächsverlauf geht verloren | Kurzzeit-Memory im State |
| Kontext wächst unbegrenzt | Sliding Window oder Summarization |
| Freigegebene Präferenzen fehlen beim nächsten Start | persistentes Memory mit Write-Regeln |
| Sessions vermischen sich | Per-User- oder Per-Projekt-Konfiguration |


In [ ]:
#@markdown ❌ Anti-Pattern: alles dauerhaft speichern { display-mode: "form" }
# Deterministische Demo: flüchtige und sensible Session-Details landen ungeprüft im persistenten Memory.

session_notizen = [
    "Dauerhaft erlaubt: Mara bevorzugt Antworten mit Kurzfazit und Quellenhinweisen.",
    "Nur Session-State: aktueller Entwurf ist noch unsicher und soll nicht wiederverwendet werden.",
    "Nicht speichern: API-Key-Fragmente, private Namen oder Rohnotizen aus einem Review.",
]

naives_langzeit_memory = []
for notiz in session_notizen:
    naives_langzeit_memory.append(notiz)

print("Naiv gespeicherte Memory-Einträge:")
for eintrag in naives_langzeit_memory:
    print(f"- {eintrag}")

assert any("Nicht speichern" in eintrag for eintrag in naives_langzeit_memory),     "Das Anti-Pattern sollte unerlaubte Informationen speichern."
print("❌ Sichtbares Scheitern: Session-State, Präferenzen und sensible Notizen werden vermischt.")


# 2 | Memory als kontrollierte Schicht
---



Genau dieses Problem löst eine getrennte Memory-Architektur. Checkpointing beantwortet, was in einer laufenden Session passiert ist. Memory-Systeme beantworten, welche Informationen nach **bewusster** Prüfung in späteren Sessions wieder auftauchen dürfen.

Für den Meeting- & Research-Briefing-Agent bedeutet das: dauerhafte Präferenzen werden kuratiert gespeichert, temporäre Arbeitsnotizen bleiben im Checkpoint, und sensible Inhalte werden nicht übernommen.


In [ ]:
import os
import sqlite3
from pathlib import Path
from typing import TypedDict, Annotated

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model
from langchain_core.messages import trim_messages, RemoveMessage, SystemMessage, HumanMessage, AIMessage

from genai_lib.model_config import WORKER
llm = init_chat_model(WORKER)


In [ ]:
#@markdown   <p><font size="4" color='green'>  Memory-Systeme</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart TB
    MEMORY[Memory-Systeme] --> SHORT[Kurzzeit-Memory]
    MEMORY --> LONG[Persistentes Memory]

    SHORT --> BUFFER["Conversation Buffer<br/>Voller Verlauf"]
    SHORT --> WINDOW["Sliding Window<br/>Letzte N Nachrichten"]
    SHORT --> SUMMAR["Summarization<br/>Komprimierter Verlauf"]

    LONG --> SEMANTIC["Semantisches Memory<br/>Vektordatenbank"]
    LONG --> ENTITY["Entity Memory<br/>Key-Value Struktur"]

    style MEMORY fill:#87CEEB,color:#000
    style SHORT  fill:#90EE90,color:#000
    style LONG   fill:#90EE90,color:#000
'''

mermaid(diagram, width=1050)

# 3 | Kurzzeit-Memory
---



Drei Strategien, die sich darin unterscheiden, wie viel Verlauf aufbewahrt wird.

## 3.1 | Conversation Buffer – Vollständiger Verlauf
---

In [ ]:
class BufferState(TypedDict):
    messages: Annotated[list, add_messages]


def _research_memory_antwort(messages: list) -> str:
    history_text = "\n".join(getattr(msg, "content", str(msg)) for msg in messages)
    aktuelle_nachricht = getattr(messages[-1], "content", str(messages[-1])) if messages else ""

    if "Welche Präferenz" in aktuelle_nachricht or "Briefing-Präferenz" in aktuelle_nachricht or "Research-Vorgaben" in aktuelle_nachricht:
        bekannte = []
        if "Mara bevorzugt Kurzfazit" in history_text or "Kurzfazit plus Quellenhinweise" in history_text:
            bekannte.append("Kurzfazit plus Quellenhinweise")
        if "RAG-Evaluation" in history_text:
            bekannte.append("Thema RAG-Evaluation")
        if "Unsicherheitsmarkierung" in history_text:
            bekannte.append("Unsicherheitsmarkierung")
        if "Out-of-Corpus" in history_text:
            bekannte.append("Out-of-Corpus-Fragen klar markieren")
        if "Max prüft Prompt-Injection" in history_text:
            bekannte.append("Prompt-Injection und Tool-Grenzen prüfen")
        if bekannte:
            return "Bekannte Research-Vorgaben: " + "; ".join(dict.fromkeys(bekannte)) + "."
        return "Keine dauerhafte Briefing-Präferenz im Session-Kontext."

    if aktuelle_nachricht.startswith("Dauerhaft erlaubt:"):
        return "Kontext notiert: freigegebene Briefing-Präferenz wurde gespeichert."
    if "Mara arbeitet am Thema RAG-Evaluation" in aktuelle_nachricht:
        return "Kontext notiert: Mara arbeitet am Thema RAG-Evaluation."
    if "Mara bevorzugt Kurzfazit" in aktuelle_nachricht:
        return "Kontext notiert: Mara bevorzugt Kurzfazit und Quellenhinweise."
    if "Max prüft Prompt-Injection" in aktuelle_nachricht:
        return "Kontext notiert: Max prüft Prompt-Injection und Tool-Grenzen."
    if "Gewünschtes Format" in aktuelle_nachricht:
        return "Kontext notiert: gewünschtes Format ist Kurzfazit plus Quellenhinweise."
    if "Out-of-Corpus" in aktuelle_nachricht:
        return "Kontext notiert: Out-of-Corpus-Fragen sollen klar markiert werden."
    return "Kontext notiert."


def buffer_chat(state: BufferState) -> BufferState:
    return {"messages": [AIMessage(content=_research_memory_antwort(state["messages"]))]}


graph = StateGraph(BufferState)
graph.add_node("chat", buffer_chat)
graph.add_edge(START, "chat")
graph.add_edge("chat", END)

buffer_app = graph.compile(checkpointer=InMemorySaver())
mprint("✅ Conversation Buffer aufgebaut")


In [ ]:
#@markdown   <p><font size="4" color='green'> Conversation Buffer</font> </br></p>

diagram = buffer_app.get_graph().draw_mermaid()
mermaid(diagram, width=400)

In [ ]:
session_cfg = {"configurable": {"thread_id": "buffer-demo"}}
run_cfg = {"run_name": "M19_BufferMemory", "tags": ["m19", "buffer"]}

for nachricht in ["Mara arbeitet am Thema RAG-Evaluation.", "Gewünschtes Format ist Kurzfazit plus Quellenhinweise.", "Welche Research-Vorgaben sind bekannt?"]:
    result = buffer_app.invoke(
        {"messages": [HumanMessage(content=nachricht)]},
        config={**session_cfg, **run_cfg}
    )
    antwort = result["messages"][-1].content
    mprint(f"**Nutzer:** {nachricht}  \n**Agent:** {antwort}")
    mprint("---")

## 3.2 | Sliding Window – Letzte N Nachrichten
---



`trim_messages()` begrenzt den Kontext auf die jüngsten Nachrichten, sobald ein Token-Limit erreicht wird.

In [ ]:
class WindowState(TypedDict):
    messages: Annotated[list, add_messages]


def window_chat(state: WindowState) -> WindowState:
    trimmed = trim_messages(
        state["messages"],
        max_tokens=2000,
        strategy="last",
        token_counter=llm,
        include_system=True,
        allow_partial=False,
    )
    return {"messages": [AIMessage(content=_research_memory_antwort(trimmed))]}


graph2 = StateGraph(WindowState)
graph2.add_node("chat", window_chat)
graph2.add_edge(START, "chat")
graph2.add_edge("chat", END)
window_app = graph2.compile(checkpointer=InMemorySaver())

# Veranschaulichung: trim_messages kürzt
nachrichten = [HumanMessage(content=f"Nachricht {i}") for i in range(1, 8)]
gekürzt = trim_messages(nachrichten, max_tokens=100, strategy="last", token_counter=llm)
mprint(f"Original: {len(nachrichten)} Nachrichten → Nach trim_messages: {len(gekürzt)} Nachrichten")


**Was passiert hier?**

1. `StateGraph(...)` — definiert den Graphen mit dem State-Schema
2. `add_node(...)` — registriert einen Knoten im Graphen
3. `add_edge(...)` — verbindet zwei Knoten mit einer festen Kante
4. `InMemorySaver()` — speichert den Graph-Zustand im RAM für Session-Persistenz
5. `compile(...)` — schließt den Graphen ab und erzeugt das ausführbare Objekt

In [ ]:
#@markdown   <p><font size="4" color='green'> Sliding Window</font> </br></p>

diagram = window_app.get_graph().draw_mermaid()
mermaid(diagram, width=400)

## 3.3 | Summarization Memory – Komprimierter Verlauf
---




Ältere Nachrichten werden vom LLM zusammengefasst statt verworfen. Der Kontext bleibt erhalten, der Token-Verbrauch wird begrenzt.

In [ ]:
class SummaryState(TypedDict):
    messages: Annotated[list, add_messages]
    summary: str


def summarize_if_needed(state: SummaryState) -> SummaryState:
    messages = state["messages"]
    if len(messages) < 6:
        return {}

    existing = state.get("summary", "")
    zum_komprimieren = messages[:-4]
    quelle = "\n".join(getattr(m, "content", str(m)) for m in zum_komprimieren)

    punkte = []
    if "RAG-Evaluation" in quelle:
        punkte.append("Thema: RAG-Evaluation für Fachartikel")
    if "Quellenhinweise" in quelle:
        punkte.append("Format: knapp mit Quellenhinweisen")
    if "Out-of-Corpus" in quelle:
        punkte.append("Grenze: Out-of-Corpus klar markieren")
    neue_zusammenfassung = "; ".join(punkte) if punkte else existing or "Keine dauerhaften Briefing-Präferenzen"

    zu_entfernen = [RemoveMessage(id=m.id) for m in zum_komprimieren]
    summary_msg = SystemMessage(content=f"Bisheriger Verlauf (komprimiert): {neue_zusammenfassung}")
    return {"messages": [summary_msg] + zu_entfernen, "summary": neue_zusammenfassung}


def summary_chat(state: SummaryState) -> SummaryState:
    messages = state.get("messages", [])
    history_text = "\n".join(getattr(m, "content", str(m)) for m in messages)
    summary = state.get("summary", "")
    recall_basis = f"{summary}\n{history_text}"
    ist_recall_frage = "Briefing-Präferenz" in recall_basis or "Research-Vorgaben" in recall_basis

    if ist_recall_frage:
        punkte = []
        if "RAG-Evaluation" in recall_basis:
            punkte.append("Thema: RAG-Evaluation für Fachartikel")
        if "Quellenhinweise" in recall_basis:
            punkte.append("Format: knapp mit Quellenhinweisen")
        if "Out-of-Corpus" in recall_basis:
            punkte.append("Grenze: Out-of-Corpus klar markieren")
        if punkte:
            antwort = "Bekannte dauerhafte Briefing-Präferenzen: " + "; ".join(dict.fromkeys(punkte)) + "."
        else:
            antwort = "Keine dauerhaften Briefing-Präferenzen im komprimierten Verlauf."
    else:
        antwort = _research_memory_antwort(messages)
    return {"messages": [AIMessage(content=antwort)]}


graph3 = StateGraph(SummaryState)
graph3.add_node("summarize", summarize_if_needed)
graph3.add_node("chat", summary_chat)
graph3.add_edge(START, "summarize")
graph3.add_edge("summarize", "chat")
graph3.add_edge("chat", END)
summary_app = graph3.compile(checkpointer=InMemorySaver())
mprint("✅ Summarization Memory aufgebaut")


**Was passiert hier?**

1. `StateGraph(...)` — definiert den Graphen mit dem State-Schema
2. `add_node(...)` — registriert einen Knoten im Graphen
3. `add_edge(...)` — verbindet zwei Knoten mit einer festen Kante
4. `InMemorySaver()` — speichert den Graph-Zustand im RAM für Session-Persistenz
5. `compile(...)` — schließt den Graphen ab und erzeugt das ausführbare Objekt

In [ ]:
#@markdown   <p><font size="4" color='green'> Summarization Memory</font> </br></p>

diagram = summary_app.get_graph().draw_mermaid()
mermaid(diagram, width=400)

In [ ]:
session_cfg = {"configurable": {"thread_id": "summary-demo"}}
run_cfg = {"run_name": "M19_SummarizationMemory", "tags": ["m19", "summarization"]}

themen = [
    "Mara untersucht RAG-Evaluation für Fachartikel.",
    "Die Antwort soll knapp sein und Quellenhinweise enthalten.",
    "Out-of-Corpus-Fragen sollen klar markiert werden.",
    "LangGraph wird für kontrollierte Research-Workflows genutzt.",
    "Welche dauerhaften Briefing-Präferenzen sind bekannt?",
]
for nachricht in themen:
    result = summary_app.invoke(
        {"messages": [HumanMessage(content=nachricht)]},
        config={**session_cfg, **run_cfg}
    )
    antwort = result["messages"][-1].content
    n = len(result["messages"])
    mprint(f"**({n} Msgs) Nutzer:** {nachricht}  \n**Agent:** {antwort[:140]}...")
    mprint("---")


# 4 | Persistentes Memory
---



Persistentes Memory überlebt das Sitzungsende und steht in zukünftigen Gesprächen zur Verfügung.

**RAG löst nur das Input-Problem**

RAG verbindet den Agenten mit einer Vektordatenbank und zieht relevante Informationen in das Kontext-Fenster —
aber RAG ist eine **Read-Only-Operation**. Es löst das Input-Problem (Kontext rein ins Modell),
nicht das Output-Problem.

Wenn ein Agent ein Python-Skript schreibt, ein Playbook erstellt oder eine Analyse produziert:
wo landet dieses Arbeitsprodukt? Ohne explizite Persistenzschicht nirgends — es verschwindet mit dem Kontext-Fenster.

| | RAG | Persistentes Memory |
|---|---|---|
| **Richtung** | Read (Daten → Agent) | Read + Write (Agent → Speicher) |
| **Löst** | Input-Problem | Input- und Output-Problem |
| **Beispiel** | Dokumente abrufen | Agenten-Output speichern und wiederverwenden |

## 4.1 | Semantisches Memory – Vektordatenbank
---

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.tools import tool
import os

MEMORY_DIR = "./semantic_memory"
os.makedirs(MEMORY_DIR, exist_ok=True)

embeddings = OpenAIEmbeddings(model=EMBEDDINGS)
memory_store = Chroma(
    collection_name="meeting_briefing_memory",
    embedding_function=embeddings,
    persist_directory=MEMORY_DIR,
)

@tool
def memory_speichern(information: str) -> str:
    """MEMORY SPEICHERN - Speichert eine freigegebene Briefing-Präferenz dauerhaft.

    Zulässig sind kuratierte Angaben wie Ausgabeformat, Themeninteresse, Quellenregel
    oder Out-of-Corpus-Verhalten. Flüchtige Rohnotizen, private Daten und Secrets
    gehören nicht in dieses Memory.
    """
    unerlaubt = ["api-key", "secret", "nicht speichern", "rohnotiz"]
    if any(marker in information.lower() for marker in unerlaubt):
        return "Nicht gespeichert: Information ist nicht für persistentes Memory geeignet."
    memory_store.add_texts([information])
    return f"Gespeichert: {information}"

@tool
def memory_abrufen(frage: str) -> str:
    """MEMORY ABRUFEN - Sucht freigegebene Briefing-Präferenzen.

    Args:
        frage: Suchanfrage zu Ausgabeformat, Themeninteresse oder Quellenregeln.
    """
    docs = memory_store.similarity_search(frage, k=3)
    if not docs:
        return "Keine relevanten Briefing-Präferenzen gefunden."
    return "\n".join(f"- {d.page_content}" for d in docs)

mprint(f"✅ Semantisches Briefing-Memory initialisiert (persistent: {MEMORY_DIR})")


In [ ]:
run_cfg = {"run_name": "M19_SemanticMemory", "tags": ["m19", "semantic-memory"]}

speicher_result = memory_speichern.invoke({
    "information": "Präferenz: Mara bevorzugt Kurzfazit, Quellenhinweis und Unsicherheitsmarkierung."
}, config=run_cfg)
mprint(f"**Speichern:** {speicher_result}")

abruf_result = memory_abrufen.invoke({
    "frage": "Welche Ausgabepräferenzen sind bekannt?"
}, config=run_cfg)
mprint(f"**Abruf:** {abruf_result}")

assert "Kurzfazit" in abruf_result and "Quellen" in abruf_result,     "❌ Semantisches Memory liefert die gespeicherte Ausgabepräferenz nicht zurück."


**Was passiert hier?**

1. `create_agent(...)` — erstellt einen Agenten mit Modell, Tools und System-Prompt

## 4.2 | Entity Memory – Key-Value für Entitäten
---

In [ ]:
from pydantic import BaseModel, Field

class Entitaet(BaseModel):
    name: str = Field(description="Name der Entität, zum Beispiel Dokument, Projekt oder Methode")
    beschreibung: str = Field(description="Kurze Beschreibung in einem Satz")
    kategorie: str = Field(description="Dokument, Projekt, Methode oder Person")

class EntitaetListe(BaseModel):
    entitaeten: list[Entitaet] = Field(default_factory=list)

class EntityState(TypedDict):
    messages: Annotated[list, add_messages]
    entity_memory: dict
    entitaeten: list[dict]

extractor = llm.with_structured_output(EntitaetListe)

FRAGE_PRÄFIXE = ("was ", "wer ", "wie ", "wo ", "wann ", "warum ", "welche", "kennst")

def entity_extraktor(state: EntityState) -> EntityState:
    letzte = state["messages"][-1].content.strip()
    if letzte.endswith("?") or letzte.lower().startswith(FRAGE_PRÄFIXE):
        return {}

    result = extractor.invoke(
        f"Extrahiere wichtige Entitäten (Dokumente, Projekte, Methoden, Personen) aus:\n{letzte}"
    )
    updated = dict(state.get("entity_memory", {}))
    entitaeten = []
    for e in result.entitaeten:
        data = e.model_dump()
        entitaeten.append(data)
        eintrag = f"[{data['kategorie']}] {data['beschreibung']}"
        if data["name"] in updated and eintrag not in updated[data["name"]]:
            updated[data["name"]] = updated[data["name"]] + "; " + eintrag
        else:
            updated[data["name"]] = eintrag
    return {"entity_memory": updated, "entitaeten": entitaeten}


def entity_chat(state: EntityState) -> EntityState:
    kontext = "\n".join(f"- {k}: {v}" for k, v in state.get("entity_memory", {}).items())
    if "Korpusstudie-01" in state["messages"][-1].content and kontext:
        antwort = "Bekannt zu Korpusstudie-01: Dokument im Projekt Meeting- & Research-Briefing-Agent; behandelt Retrieval-Qualität; Quellenhinweise werden genutzt."
    else:
        antwort = "Entity Memory aktualisiert."
    return {"messages": [AIMessage(content=antwort)]}


graph4 = StateGraph(EntityState)
graph4.add_node("extraktor", entity_extraktor)
graph4.add_node("chat", entity_chat)
graph4.add_edge(START, "extraktor")
graph4.add_edge("extraktor", "chat")
graph4.add_edge("chat", END)
entity_app = graph4.compile(checkpointer=InMemorySaver())
mprint("✅ Entity Memory für Research-Entitäten aufgebaut")


**Was passiert hier?**

1. `StateGraph(...)` — definiert den Graphen mit dem State-Schema
2. `add_node(...)` — registriert einen Knoten im Graphen
3. `add_edge(...)` — verbindet zwei Knoten mit einer festen Kante
4. `InMemorySaver()` — speichert den Graph-Zustand im RAM für Session-Persistenz
5. `compile(...)` — schließt den Graphen ab und erzeugt das ausführbare Objekt
6. `with_structured_output(...)` — bindet das Pydantic-Schema ans Modell und erzwingt strukturierte Ausgabe

In [ ]:
#@markdown   <p><font size="4" color='green'>Entity Memory</font> </br></p>

diagram = entity_app.get_graph().draw_mermaid()
mermaid(diagram, width=400)

In [ ]:
session_cfg = {"configurable": {"thread_id": "entity-demo"}}
run_cfg = {"run_name": "M19_EntityMemory", "tags": ["m19", "entity-memory"]}

nachrichten = [
    "Mara analysiert Dokument Korpusstudie-01 im Projekt Meeting- & Research-Briefing-Agent.",
    "Korpusstudie-01 behandelt Retrieval-Qualität und das Projekt Meeting- & Research-Briefing-Agent nutzt Quellenhinweise.",
    "Was ist über Korpusstudie-01 bekannt?",
]
for msg in nachrichten:
    result = entity_app.invoke(
        {"messages": [HumanMessage(content=msg)]},
        config={**session_cfg, **run_cfg}
    )
    mem = result.get("entity_memory", {})
    antwort = result["messages"][-1].content
    mprint(f"**Nutzer:** {msg}")
    if mem:
        mprint(f"*Entity Memory: {mem}*")
    mprint(f"**Agent:** {antwort}")
    mprint("---")

## 4.3 | Persistenter Checkpointer – SqliteSaver
---



**Problem:** `InMemorySaver` speichert den Graphzustand nur im RAM – nach einem Notebook-Neustart ist der Zustand weg.

**Lösung:** `SqliteSaver` schreibt jeden Checkpoint in eine SQLite-Datei. Der Graph-Zustand überlebt Neustarts und ist für alle Threads dauerhaft verfügbar.

| | `InMemorySaver` | `SqliteSaver` |
|--|--------------|---------------|
| **Speicher** | RAM | SQLite-Datei auf Disk |
| **Neustart** | ❌ Zustand verloren | ✅ Zustand erhalten |
| **Einsatz** | Demos, Entwicklung | Staging, Production |
| **Setup** | `InMemorySaver()` | `SqliteSaver(conn)` |

In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

SESSION_DB = "./agent_sessions.db"
for suffix in ("", "-wal", "-shm"):
    db_file = Path(f"{SESSION_DB}{suffix}")
    if db_file.exists():
        db_file.unlink()

conn = sqlite3.connect(SESSION_DB, check_same_thread=False)
persistent_checkpointer = SqliteSaver(conn)

persistent_graph = StateGraph(BufferState)
persistent_graph.add_node("chat", buffer_chat)
persistent_graph.add_edge(START, "chat")
persistent_graph.add_edge("chat", END)
persistent_app = persistent_graph.compile(checkpointer=persistent_checkpointer)
mprint(f"✅ Persistenter Graph mit SqliteSaver aufgebaut ({SESSION_DB})")


In [ ]:
session_cfg = {"configurable": {"thread_id": "research-mara-sqlite"}}
run_cfg = {"run_name": "M19_SqliteSaver", "tags": ["m19", "sqlite"]}

result = persistent_app.invoke(
    {"messages": [HumanMessage(content="Dauerhaft erlaubt: Mara arbeitet am Thema RAG-Evaluation.")]},
    config={**session_cfg, **run_cfg}
)
mprint(f"**Agent (Runde 1):** {result['messages'][-1].content}")

result = persistent_app.invoke(
    {"messages": [HumanMessage(content="Welche dauerhaften Briefing-Präferenzen sind bekannt?")]},
    config={**session_cfg, **run_cfg}
)
antwort_sqlite = result["messages"][-1].content
assert "RAG-Evaluation" in antwort_sqlite, "❌ SQLite-Checkpoint rekonstruiert das gespeicherte Research-Thema nicht."
mprint(f"**Agent (Runde 2):** {antwort_sqlite}")

mprint(f"\n*Zustand gespeichert in {SESSION_DB} – bleibt nach Notebook-Neustart erhalten.*")


In [ ]:
#@markdown   <p><font size="4" color='green'>  Persistenter Checkpointer (SqliteSaver)</font> </br></p>

diagram = persistent_app.get_graph().draw_mermaid()
mermaid(diagram, width=400)

# 5 | Per-User Memory
---



In Multi-User-Systemen trennt eine eindeutige `thread_id` pro Nutzer den Kontext vollständig.

In [ ]:
def get_config(user_id: str) -> dict:
    return {"configurable": {"thread_id": f"research-user-{user_id}"}}

run_cfg = {"run_name": "M19_PerUser", "tags": ["m19", "per-user"]}
session_cfg_mara = get_config("mara")
session_cfg_max = get_config("max")

buffer_app.invoke(
    {"messages": [HumanMessage(content="Mara bevorzugt Kurzfazit und Quellenhinweise.")]},
    config={**session_cfg_mara, **run_cfg}
)
buffer_app.invoke(
    {"messages": [HumanMessage(content="Max prüft Prompt-Injection und Tool-Grenzen.")]},
    config={**session_cfg_max, **run_cfg}
)

for name, cfg in [("Mara", session_cfg_mara), ("Max", session_cfg_max)]:
    result = buffer_app.invoke(
        {"messages": [HumanMessage(content="Welche Präferenz ist in dieser Session bekannt?")]},
        config={**cfg, **run_cfg}
    )
    mprint(f"**{name} fragt:** Welche Präferenz ist bekannt?  \n**Agent:** {result['messages'][-1].content}")


---
**Zusammenfassung**

| Memory-Typ | Implementierung | Persistenz | Einsatz |
|-----------|----------------|-----------|---------|
| **Conversation Buffer** | `add_messages` Reducer | ❌ RAM | Kurze Gespräche, einfache Demos |
| **Sliding Window** | `trim_messages()` | ❌ RAM | Lange Gespräche, Token-Budget begrenzen |
| **Summarization** | `RemoveMessage` + LLM | ❌ RAM | Multi-Turn mit wichtigem Kontext |
| **Semantisches Memory** | ChromaDB + `persist_directory` | ✅ Disk | Nutzerpräferenzen über Sessions |
| **Entity Memory** | `with_structured_output` + Dict | ❌ RAM | CRM-Agenten, Support-Systeme |
| **Persistenter Checkpointer** | `SqliteSaver` | ✅ Disk | Staging, Production, Multi-User |
| **Per-User Memory** | Thread-IDs im Checkpointer | je Checkpointer | Multi-User-Anwendungen |

**Faustregel:**
- **Kurzzeit (Demo/Dev)** → `InMemorySaver` + Conversation Buffer oder Sliding Window
- **Langzeit (Semantic)** → `ChromaDB` mit `persist_directory`
- **Langzeit (Sessions)** → `SqliteSaver` als Checkpointer
- **Per-User** → eindeutige Thread-IDs + persistenter Checkpointer

Verwandte Module: *Checkpointing & Sessions*, *Human-in-the-Loop*, *Multi-Agent Patterns*

In [ ]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M19-Memory-Systeme", limit=3, show_steps=True)

# A | Aufgaben
---



<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen. Eigene Varianten sind willkommen, solange die Grenze zwischen Session-State und dauerhaftem Briefing-Memory sichtbar bleibt.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf generative KI als Unterstützung beim Lernen und Entwickeln genutzt werden. Bei einer Blockade kann zum Beispiel Gemini in Google Colab helfen, Fehlermeldungen zu verstehen, Ideen für Teilschritte zu prüfen oder Code-Varianten zu vergleichen.
> <br>Der Schwerpunkt bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


**Grundlagen**
- Eine Allowlist-Funktion `darf_dauerhaft_speichern(text)` definieren.
- Zulässig sind nur Briefing-Präferenzen wie Ausgabeformat, Themeninteresse, Quellenregel oder Out-of-Corpus-Verhalten.
- Sensible Inhalte, Rohnotizen und Session-Entwürfe werden abgelehnt.

**✅ Erledigt wenn:** Erlaubte Briefing-Präferenzen werden akzeptiert; verbotene Inhalte werden abgelehnt.


In [ ]:
# Grundlagen: Write-Regel für persistentes Memory
def darf_dauerhaft_speichern(text: str) -> bool:
    text_l = text.lower()
    erlaubte_marker = ["präferenz", "ausgabeformat", "quellen", "out-of-corpus", "themeninteresse"]
    verbotene_marker = ["api-key", "secret", "rohnotiz", "nicht speichern", "privat"]
    return any(marker in text_l for marker in erlaubte_marker) and not any(marker in text_l for marker in verbotene_marker)

beispiel_erlaubt = "Präferenz: Mara bevorzugt Kurzfazit mit Quellen."
beispiel_verboten = "Nicht speichern: Rohnotiz mit API-Key-Fragment."
print("Erlaubt:", darf_dauerhaft_speichern(beispiel_erlaubt))
print("Verboten:", darf_dauerhaft_speichern(beispiel_verboten))


**Aufbau**
- Ein kleines `research_memory` als Key-Value-Store anlegen.
- Nur Informationen speichern, die `darf_dauerhaft_speichern(...)` passieren.
- Eine Abruffunktion für bekannte Briefing-Präferenzen bereitstellen.

**✅ Erledigt wenn:** Der Store enthält erlaubte Präferenzen und keine verbotenen Rohnotizen.


In [ ]:
# Aufbau: kuratiertes Briefing-Memory
research_memory = {}

def research_memory_speichern(key: str, text: str) -> str:
    if not darf_dauerhaft_speichern(text):
        return "Nicht gespeichert"
    research_memory[key] = text
    return "Gespeichert"

def research_memory_abrufen(key: str) -> str:
    return research_memory.get(key, "Keine gespeicherte Präferenz")

print(research_memory_speichern("mara_briefing_format", "Präferenz: Kurzfazit mit Quellen und Unsicherheitsmarkierung."))
print(research_memory_speichern("session_rohnotiz", "Nicht speichern: Rohnotiz aus aktuellem Review."))
print(research_memory_abrufen("mara_briefing_format"))


**Vertiefung**
- Per-User-Isolation ergänzen: `research_memory_by_user[user_id][key]`.
- Zwei Nutzer mit unterschiedlichen Präferenzen speichern.
- Prüfen, dass Abrufe nicht zwischen Nutzern vermischt werden.

**✅ Erledigt wenn:** Mara und Max erhalten getrennte Memory-Einträge.


In [ ]:
# Vertiefung: Per-User-Memory
research_memory_by_user = {}

def user_memory_speichern(user_id: str, key: str, text: str) -> str:
    if not darf_dauerhaft_speichern(text):
        return "Nicht gespeichert"
    research_memory_by_user.setdefault(user_id, {})[key] = text
    return "Gespeichert"

def user_memory_abrufen(user_id: str, key: str) -> str:
    return research_memory_by_user.get(user_id, {}).get(key, "Keine gespeicherte Präferenz")

user_memory_speichern("mara", "format", "Präferenz: Kurzfazit mit Quellen.")
user_memory_speichern("max", "format", "Präferenz: Security-Hinweis mit Tool-Grenzen.")
print("Mara:", user_memory_abrufen("mara", "format"))
print("Max:", user_memory_abrufen("max", "format"))


**Was passiert hier?**

1. Grundlagen definieren eine Write-Regel für dauerhaftes Memory.
2. Aufbau speichert nur kuratierte Briefing-Präferenzen.
3. Vertiefung trennt Memory pro Nutzer oder Rolle.
4. Die Selfchecks prüfen nicht nur Variablennamen, sondern den tatsächlichen Speicherinhalt.


# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [Memory-Systeme](https://ralf-42.github.io/Agenten/04-agenten-implementierung/ablauf-zustand/memory-systeme.html)
- [Context Engineering](https://ralf-42.github.io/Agenten/04-agenten-implementierung/kontext-wissen/context-engineering.html)
- [Checkpointing & Persistenz](https://ralf-42.github.io/Agenten/04-agenten-implementierung/ablauf-zustand/checkpointing-persistenz.html)
- [State Management](https://ralf-42.github.io/Agenten/04-agenten-implementierung/ablauf-zustand/state-management.html)
- [Checkliste Agentensystem](https://ralf-42.github.io/Agenten/04-agenten-implementierung/checkliste-agentensystem.html)
